In [1]:
import re
import fitz  # PyMuPDF
import pandas as pd
import streamlit as st
import io

def extract_text_from_pdf(pdf_file):
    """Trích xuất toàn bộ text từ file PDF (file object từ Streamlit)."""
    text = ""
    with fitz.open(stream=pdf_file.read(), filetype="pdf") as pdf:
        for page in pdf:
            text += page.get_text("text")
    return text

with open(r"E:\D\HOCDIBANTRE\AnhSa\20251019_SGN_CHNLASC0483158_JP_PARTA_1.pdf", "rb") as f:
    text = extract_text_from_pdf(f)

In [33]:
import pdfplumber
import fitz

# # fitz -> lấy vị trí và layout
# doc = fitz.open(r"E:\D\HOCDIBANTRE\AnhSa\20251019_SGN_CHNLASC0483158_JP_PARTA_1.pdf")

# pdfplumber -> lấy bảng, text có cấu trúc
with pdfplumber.open(r"E:\D\HOCDIBANTRE\AnhSa\20251103_SGN_CHNLASC0486289_JP_PARTA.pdf") as pdf:
    text = "\n".join(page.extract_text() or "" for page in pdf.pages)


In [37]:
import re
import pandas as pd

def find_text_before_keyword(text, keyword):
    """
    Extract the number (or text) immediately before the given keyword.
    Example:
        '31 Cartons of Footwear Division of Goods' -> '31'
        '25 Cartons of Apparel Division goods'     -> '25'
    Supports:
        - Any single word between 'Cartons of' and 'Division'
        - Both 'Division of goods' and 'Division goods'
        - Any capitalization of 'Goods' or 'goods'
    """
    # Nếu keyword có chứa 'Cartons of', dùng regex linh hoạt
    if re.search(r"Cartons of", keyword, re.IGNORECASE):
        # Cho phép có hoặc không có 'of' trước 'Goods'
        pattern = r"(\d+)\s+(?=Cartons of \w+ Division(?: of)? [Gg]oods)"
    else:
        # Trường hợp bình thường
        pattern = rf"(\d+)\s+(?={re.escape(keyword)})"

    match = re.search(pattern, text, re.IGNORECASE)
    return match.group(1) if match else None

def find_text_before_keyword_2(text, keyword):
    """
    Extract the number (or text) immediately before the given keyword.
    Example: '31 Cartons of Footwear Division Goods' -> '31'
    """
    pattern = rf'(\d+)\s+(?={re.escape(keyword)})'
    match = re.search(pattern, text)
    return match.group(1) if match else None

def find_text_after_keyword(text, keyword, num_chars=50):
    """
    Tìm đoạn văn bản sau một keyword. 
    Nếu cùng dòng không có nội dung, lấy nội dung ở dòng kế tiếp.
    """
    # 1️⃣ Tìm trên cùng dòng
    pattern_same_line = re.escape(keyword) + r"[ \t]*(.{1," + str(num_chars) + r"})"
    # print("Searching for keyword pattern:", pattern_same_line)
    match = re.search(pattern_same_line, text)
    if match:
        result = match.group(1).strip()
        if result:
            return result

    # 2️⃣ Nếu không có, tìm ở dòng kế tiếp
    pattern_next_line = re.escape(keyword) + r"\s*\n\s*(.{1," + str(num_chars) + r"})"
    match = re.search(pattern_next_line, text)
    if match:
        return match.group(1).strip()

    return None

def find_text_after_style(text, keyword, num_chars=50):
    """
    Tìm nội dung sau một keyword.
    Keyword phải có ít nhất 1 khoảng trắng sau từ chính, hoặc dấu ':' / '#:'.
    Hỗ trợ các biến thể: 'MATERIAL ', 'MATERIAL:', 'MATERIAL #:'.
    """
    # Bắt buộc có ít nhất 1 space sau từ, có thể có ':' hoặc '#:' theo sau
    keyword_pattern = re.escape(keyword.rstrip(".:")) + r"(?:[.:]+)?(?:\s+(?:#?:)?)"

    # 1️⃣ Tìm trên cùng dòng
    pattern_same_line = keyword_pattern + r"(.{1," + str(num_chars) + r"})"
    match = re.search(pattern_same_line, text, re.IGNORECASE)
    if match:
        result = match.group(1).strip()
        if result:
            return result

    # 2️⃣ Tìm ở dòng kế tiếp nếu không có gì trên cùng dòng
    pattern_next_line = keyword_pattern + r"\s*\n\s*(.{1," + str(num_chars) + r"})"
    match = re.search(pattern_next_line, text, re.IGNORECASE)
    if match:
        return match.group(1).strip()

    return None

def split_by_bill_names(text, bill_names):
    """
    Tách text thành các phần tương ứng với bill_names (theo thứ tự trong danh sách).
    Mỗi bill_name chỉ xuất hiện 1 lần, lấy nội dung từ bill_i đến bill_(i+1).
    """
    sections = []
    text_lower = text.lower()
    positions = []
    for b in bill_names:
        # print("Checking bill name:", b)
        if b == "WAYBILL":
            idx = text_lower.find(b.lower())
            positions.append((idx, b))
        else:
        # Regex: tìm tất cả vị trí 'Invoice 1' KHÔNG có '(continue)' ngay sau
            for match in re.finditer(rf'\b{re.escape(b)}\b(?!\s*\(continued\))', text_lower, re.IGNORECASE):
                # print(f"Found '{b}' at position {match.start()}")
                positions.append((match.start(), b))

    
    positions.sort(key=lambda x: x[0])
    
    # Tách nội dung giữa các bill
    for i, (start_idx, bill) in enumerate(positions):
        end_idx = positions[i+1][0] if i + 1 < len(positions) else len(text)
        content = text[start_idx + len(bill): end_idx].strip()
        sections.append((bill, content))
    return sections

def split_waybill_blocks(text):
    pattern = r"(\d+\s*CTN\s*//.*?\*{34,})"
    blocks = re.findall(pattern, text, flags=re.DOTALL | re.IGNORECASE)
    return blocks

def extract_bill_sections(text, bill_names, keyword_dict, verbose=False):
    """
    Extract info for each bill section based on grouped keywords.
    Each main key in keyword_dict will become a column name.
    """
    sections = split_by_bill_names(text, bill_names)
    results = []

    for bill_name, content in sections:
        # if verbose and bill_name == "Factory Packing List":
        #     print(f"Processing section: {bill_name}")
        #     print(f"Snippet: {content}...")
        #     print("-" * 50)
            
        if bill_name == "WAYBILL":
            blocks = split_waybill_blocks(content)
            for i, block in enumerate(blocks):
                entry = {"Bill Name": f"WAYBILL_{i+1}"}
                # print("Processing WAYBILL block:", block)
                for key, kw_list in keyword_dict.items():
                    if not kw_list:
                        entry[key] = "⚠️ Not Found"
                        continue
                    idx = bill_names.index(bill_name)
                    kw = kw_list[idx]
                    if kw == "":
                        entry[key] = None
                        continue
                    if kw == "Cartons of Footwear Division Goods" or kw == "Cartons of Footwear Division of goods":
                        cont = find_text_before_keyword(block, kw) 
                        if cont is None:
                            cont = find_text_before_keyword_2(block, "CNTS OF NIKE FOOTWEAR GOODS")  
                        if cont is None:
                            cont = find_text_before_keyword_2(block, "CARTONS")
                        if cont is None:
                            cont = find_text_before_keyword_2(block, "CTN")
                    elif kw == "PO-Item:":
                        cont = find_text_after_keyword(block, kw)
                        if cont is None:
                            kw = "PO: "
                            cont = find_text_after_keyword(block, kw)
                        if cont is None:
                            kw = "PO:"
                            cont = find_text_after_keyword(block, kw)
                    elif kw == "Invoice#:":
                        cont = find_text_after_keyword(block, kw)
                        if cont is None:
                            kw = "Invoice number:"
                            cont = find_text_after_keyword(block, kw)
                        if cont is None:
                            kw = "INVOICE NUMBER:"
                            cont = find_text_after_keyword(block, kw)
                    elif kw == "Material:":
                        cont = find_text_after_keyword(block, kw)
                        if cont is None:
                            kw = "MATERIAL:"
                            cont = find_text_after_keyword(block, kw)
                    elif kw == "Qty:":
                        cont = find_text_after_keyword(block, kw)
                        if cont is None:
                            kw = "Quantity:"
                            cont = find_text_after_keyword(block, kw)
                        if cont is None:
                            kw = "QUANTITY:"
                            cont = find_text_after_keyword(block, kw)
                    else:
                        cont = find_text_after_keyword(block, kw)

                    if cont:
                        entry[key] = cont
                    else:
                        entry[key] = None
            
                results.append(entry)
            continue 
         
        entry = {"Bill Name": bill_name}
        for key, kw_list in keyword_dict.items():
            if not kw_list:
                entry[key] = "⚠️ Not Found"
                continue
            idx = bill_names.index(bill_name)
            kw = kw_list[idx]
            if kw == "":
                entry[key] = None
                continue
            if kw == "Cartons of Footwear Division Goods" or kw == "Cartons of Footwear Division of goods":
                cont = find_text_before_keyword(content, kw) 
                if cont is None:
                    cont = find_text_before_keyword_2(content, "CNTS OF NIKE FOOTWEAR GOODS")  
                if cont is None:
                    cont = find_text_before_keyword_2(content, "CARTONS")
                if cont is None:
                    cont = find_text_after_keyword(content, "Total Cartons: ")
                    print(content)
            elif kw == "MATERIAL:" or kw == "INVOICE NO":      
                cont = find_text_after_style(content, kw)
            elif kw == "P.O. #:":
                cont = find_text_after_keyword(content, kw)
                if cont is None:
                    kw = "PO#:"
                    cont = find_text_after_keyword(content, kw)
            elif kw == "PO-Item:":
                cont = find_text_after_keyword(content, kw)
                if cont is None:
                    kw = "PO: "
                    cont = find_text_after_keyword(content, kw)
                    print(cont)
            elif kw == "ITEM :":
                cont = find_text_after_keyword(content, kw)
                if cont is None:
                    kw = "PO LINE ITEM SEQ. #:"
                    cont = find_text_after_keyword(content, kw)
                if cont is None:
                    kw = "PO Line Item Seq. #:"
                    cont = find_text_after_keyword(content, kw)
            elif kw == "Invoice#:":
                cont = find_text_after_keyword(content, kw)
                if cont is None:
                    kw = "Invoice number:"
                    cont = find_text_after_keyword(content, kw)
            else:
                cont = find_text_after_keyword(content, kw)

            if cont:
                entry[key] = cont
            else:
                entry[key] = None

        results.append(entry)

    return pd.DataFrame(results)

bill_names = [
    "WAYBILL",
    "Trading Company Commercial Invoice",
    "Factory Commercial Invoice",
    "Factory Packing List",
    "MULTIPLE COUNTRY OF ORIGIN DECLARATION",
    "Japan Customs Form"
]

keywords = {
    "INV": ["Invoice#:", "Reference Invoice #:", "Invoice Number:", "Invoice Number.:", "INVOICE NO", ""],
    "Total weight": ["", "Total Gross Weight:", "Total Gross Weight:", "Total Gross Kgs:", "",""],
    "PO": ["PO-Item:", "PO#:", "Reference PO#:", "Reference PO#:", "P.O. #:", ""],
    "PO line": ["", "PO Line Item Seq.#: ", "PO Line Item Seq. #:", "Item Seq.:", "ITEM :", ""],
    "Style": ["Material:", "Material#:", "Material #:", "Material:", "MATERIAL:", ""],
    "total carton": ["Cartons of Footwear Division of goods", "Cartons of Footwear Division Goods",  "Cartons of Footwear Division Goods", "Cartons of Footwear Division Goods", "Cartons of Footwear Division Goods", ""],
    "total quantity": ["Qty:", "Total Invoice ", "Total Invoice Quantity:", "", "",""]
}

df = extract_bill_sections(text, bill_names, keywords, verbose=True)
for i in range(len(df)):
    if "-" in str(df.loc[i, "PO"]):
        parts = df.loc[i, "PO"].split("-")
        df.loc[i, "PO"] = parts[0]
        df.loc[i, "PO line"] = parts[1] if len(parts) > 1 else None
    df["PO"] = df["PO"].astype(str).str.split(",").str[0]
    df["PO"] = df["PO"].astype(str).str.split(" ").str[0]
    df["PO line"] = df["PO line"].astype(str).str.split(",").str[0]
    df["Style"] = df["Style"].astype(str).str.split(",").str[0]
    df["Style"] = df["Style"].astype(str).str.split(" ").str[0]
    df["PO line"] = df["PO line"].astype(str).str.split(" ").str[0]
    df["INV"] = df["INV"].astype(str).str.split(" ").str[0]
    df["total quantity"] = df["total quantity"].astype(str).str.split(" ").str[0]
    df["total carton"] = df["total carton"].astype(str).str.split("  ").str[0]
    df["total carton"] = df["total carton"].astype(str).str.split(" ").str[0]
    df["total volume"] = None
df


seller: Manufacturing Factory:
ECLAT TEXTILE CO LTD ECLAT TEXTILE CO., LTD (VIETNAM).
NO. 738, ZHONGYANG RD., LOT 1, ROAD 5A, NHON TRACH II INDUSTRIAL ZONE,
XINZHUANG DIST.,NEW TAIPEI CITY 242 NHON TRACH COMMUNE,
TAIWAN DONG NAI PROVINCE, VIETNAM
1085
Tokyo, Tk Japan
Invoice Number.: VA25100949 AFS Category: 01000 MSR: No
Date: October 23, 2025 Material: DV9832-010 Total Gross Kgs: 49.65
PO# : 8501954045 Desc: AS M NK DF PRIMARY STMT SS Total Net Kgs: 44.26
Reference PO#: 3503867261 Plant: 1085 Total CBM: 0.30
Item Seq.: 00060 Customer Ship To: #: Total Cartons: 6 Country Of Origin:
Purchase Doc Date: 20250716 SP2026 Customer SO-Line #: Total Units: 228 Vietnam
Factory Code: CVT Customer PO #:
Manufacturer ID: 004719849 Buy Group: FIRST QUALITY Booking Number: VB25101309225879
To: Tokyo, Tk Japan Lot Numbers:
Carton Range Number of Ctn. Carton Contents -- Quantity by Size
From To Units Cartons Type S M L XL 2XL
3270601 3270601 32 1 Z4 32
3270602 3270602 62 1 Z6 62
3270603 3270603 66 1 

,Bill Name,INV,Total weight,PO,PO line,Style,total carton,total quantity,total volume
0,WAYBILL_1,VE25100884,None,3503856395,00080/90,IM1763-236/IM1763-502,40,530,None
1,WAYBILL_2,VB25100344,None,3503856395,00070,IM1761-100,4,60,None
2,WAYBILL_3,VA25100948,None,3503849677,00050,IM1764-502,5,60,None
3,WAYBILL_4,VA25100950,None,3503867261,00070,DV9832-098,5,124,None
4,WAYBILL_5,VA25100949,None,3503867261,00060,DV9832-010,6,228,None
5,WAYBILL_6,VB25100345,None,3503867266,00030,HV3586-502,8,89,None
6,WAYBILL_7,VB25100343,None,3503849676,00030,IM1761-502,4,60,None
7,Trading Company Commercial Invoice,VA25100949,49.65 Kgs,3503867261,00060,DV9832-010,6,228,None
8,Factory Commercial Invoice,VA25100949,49.65Kgs,3503867261,00060,DV9832-010,6,228,None
9,Factory Packing List,VA25100949,49.65,3503867261,00060,DV9832-010,6,None,None


In [4]:
keywords = {
    "INV": ["Invoice#:", "Invoice Number:", "Invoice Number.:", "INVOICE NO."],
    "Total weight": ["Total Gross Kgs:"],
    "total volume": [],
    "PO": ["PO-Item:", "P.O.#:", "PO#:", "PO Number:"],
    "PO line": ["ITEM:", "PO Line No:", "PO Line#:"],
    "Style": ["Material:", "Material No:", "Material#:", "MATERIAL"],
    "total carton": ["Total Carton:", "Total Cartons:", "Total No. of Cartons:"],
    "total quantity": ["Total Quantity:", "Total Qty:", "Qty:"]
}

In [31]:
import re

# Regex: match MATERIAL khi có ít nhất 1 khoảng trắng hoặc : hoặc #:
pattern = r"MATERIAL(?:\s+(?:#?:)?)?"

test_strings = [
    "MATERIAL",          # không match
    "MATERIAL ",         # match
    "MATERIAL: 12345",   # match
    "MATERIAL #: 54321", # match
    "MATERIAL#:",        # không match vì không có khoảng trắng
    "MATERIAL ABC"       # match
]

for s in test_strings:
    match = re.search(pattern, s)
    print(f"{s!r:20} -> {'MATCH' if match else 'NO MATCH'}")


'MATERIAL'           -> MATCH
'MATERIAL '          -> MATCH
'MATERIAL: 12345'    -> MATCH
'MATERIAL #: 54321'  -> MATCH
'MATERIAL#:'         -> MATCH
'MATERIAL ABC'       -> MATCH
